# QuantumPINNs — Combined Benchmark

This notebook runs a side-by-side comparison of PINN performance across all supported quantum problems:

| Problem | Equation | Key Metric |
|---|---|---|
| Harmonic Oscillator | TISE, harmonic $V$ | Relative L² error vs. analytic |
| Schrödinger (1D, TDSE) | TDSE, harmonic $V$ | PDE residual norm |
| Anharmonic Oscillator | TISE, quartic $V$ | Relative L² vs. numerical reference |
| Hamiltonian Dynamics | Hamilton's eqs | Energy conservation error |

Results are exported to `outputs/benchmark_summary.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import csv
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('dark_background')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

FAST_MODE = True   # Set False for publication-quality results (more epochs)

In [ ]:
# ── Run each problem via src.train for a small number of epochs ────────────
import subprocess

EPOCHS = 3000 if FAST_MODE else 10000
COL    = 1500 if FAST_MODE else 4000
results = []

for problem in ['harmonic_oscillator', 'schrodinger', 'anharmonic', 'hamiltonian']:
    model_path = f'../outputs/{problem}_bench.pt'
    t0 = time.time()
    cmd = [
        'python', '-m', 'src.train',
        '--problem', problem,
        '--epochs', str(EPOCHS),
        '--collocation', str(COL),
        '--model-path', model_path,
        '--print-every', str(EPOCHS),
        '--cpu',
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd='..')
    elapsed = time.time() - t0

    # Parse final total loss from stdout
    final_loss = None
    for line in proc.stdout.splitlines():
        if 'total=' in line:
            try:
                final_loss = float(line.split('total=')[1].split()[0])
            except Exception:
                pass

    results.append({
        'problem': problem,
        'final_loss': final_loss,
        'elapsed_s': round(elapsed, 1),
        'epochs': EPOCHS,
        'collocation': COL,
    })
    print(f'{problem:30s}  loss={final_loss}  time={elapsed:.1f}s')

In [ ]:
# ── Save benchmark summary ────────────────────────────────────────────────
os.makedirs('../outputs', exist_ok=True)
out_path = '../outputs/benchmark_summary.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['problem', 'final_loss', 'elapsed_s', 'epochs', 'collocation'])
    writer.writeheader()
    writer.writerows(results)
print(f'Saved → {out_path}')

# ── Bar chart ─────────────────────────────────────────────────────────────
problems  = [r['problem'].replace('_', '\n') for r in results]
losses    = [r['final_loss'] or 0 for r in results]

fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0d1117')
ax.set_facecolor('#161b22')
colors = ['#58a6ff', '#3fb950', '#d2a8ff', '#ffa657']
bars = ax.bar(problems, losses, color=colors, edgecolor='#30363d', linewidth=0.8)
ax.set_yscale('log')
ax.set_ylabel('Final Total Loss (log scale)', color='#8b949e')
ax.set_title(f'PINN Benchmark — {EPOCHS} epochs, {COL} collocation pts', color='#e6edf3')
ax.tick_params(colors='#8b949e')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
for bar, loss in zip(bars, losses):
    if loss:
        ax.text(bar.get_x() + bar.get_width()/2, loss * 1.3,
                f'{loss:.2e}', ha='center', va='bottom', color='#e6edf3', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/benchmark_bar.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/benchmark_bar.png')